In [ ]:
from pathlib import Path
import re
import shutil
import csv
from collections import defaultdict
from itertools import product

# =========================
# CONFIG
# =========================

PROJECT_ROOT = Path(".")
DEST_ROOT = Path("organized_models")

COPY_INSTEAD_OF_MOVE = True   # start True for safety
DRY_RUN = True               # set True to preview without copying

SEARCH_DIRS = [
    "recent_analysis",
    "recent_analysis_split_na",
    "recent_analysis_unet",
    "recent_analysis_unet_na",
    "recent_analysis_prof_unet",
    "official_darcy",
    "baseline_full",
    "data_amount_tests",
]

MODEL_ALIASES = {
    "prof_unet": "unetfixed",
    "unetFixed": "unetfixed",
    "unetfixed": "unetfixed",
    "splitnet_attn": "splitnet_attn",
    "attn_unet": "attn_unet",
    "splitnet": "splitnet",
    "unet": "unet",
}

MODEL_TYPES = [
    "unet",
    "attn_unet",
    "splitnet",
    "splitnet_attn",
    "unetfixed",
]

DATASET_MODES = [
    "fixed",
    "border",
]

TRAINING_MODES = [
    "physics_limited",
    "baseline_full",
]

DARCY_WEIGHTS = [
    "0p0",
    "0p001",
    "0p01",
    "0p1",
    "1p0",
    "5p0",
    "10p0",
]

FILE_KINDS = [
    "best_state",
    "final_state",
    "history",
    "summary",
]

EXTENSIONS = {
    "best_state": ".pt",
    "final_state": ".pt",
    "history": ".csv",
    "summary": ".json",
}


# =========================
# PARSING HELPERS
# =========================

def normalize_model(name: str) -> str:
    return MODEL_ALIASES.get(name, name)


def infer_experiment_group(path: Path) -> str:
    s = str(path)

    if "official_darcy" in s:
        return "official_darcy"
    if "baseline_full" in s:
        return "baseline_full"
    if "data_amount_tests" in s:
        return "data_amount_tests"
    if "recent_analysis" in s:
        return "quick_sweeps"

    return "unknown"


def infer_density(path: Path) -> str:
    """
    Your filenames may not always contain dense/thin.
    This tries to infer it when possible.
    """
    s = str(path).lower()

    if "thin" in s:
        return "thin"
    if "dense" in s:
        return "dense"

    # Most of your newer loader code uses Dense datasets
    return "unknown"


def parse_file(path: Path):
    """
    Attempts to parse model metadata from filename/path.
    Returns dict or None.
    """

    name = path.name

    if not any(name.endswith(ext) for ext in [".pt", ".csv", ".json"]):
        return None

    if not any(k in name for k in FILE_KINDS):
        return None

    group = infer_experiment_group(path)
    density = infer_density(path)

    dataset_mode = None
    for d in DATASET_MODES + ["border_pressure"]:
        if name.startswith(d + "_") or f"/{d}/" in str(path):
            dataset_mode = d
            break

    if dataset_mode is None:
        return None

    training_mode = None
    if "physics_limited" in name:
        training_mode = "physics_limited"
    elif "baseline_full" in name:
        training_mode = "baseline_full"
    else:
        return None

    model_type = None
    for m in sorted(MODEL_ALIASES.keys(), key=len, reverse=True):
        if m in name:
            model_type = normalize_model(m)
            break

    if model_type is None:
        return None

    darcy_weight = None
    if "nodarcy" in name:
        darcy_weight = "0p0"
    else:
        match = re.search(r"darcy_([0-9]+p[0-9]+)", name)
        if match:
            darcy_weight = match.group(1)

    sim_count = None
    match = re.search(r"sims_([0-9]+)", name)
    if match:
        sim_count = int(match.group(1))

    file_kind = None
    for k in FILE_KINDS:
        if k in name:
            file_kind = k
            break

    return {
        "source_path": path,
        "group": group,
        "density": density,
        "dataset_mode": dataset_mode,
        "training_mode": training_mode,
        "model_type": model_type,
        "darcy_weight": darcy_weight,
        "sim_count": sim_count,
        "file_kind": file_kind,
        "extension": path.suffix,
    }


# =========================
# PART 1: ORGANIZE MODELS
# =========================

def find_model_files():
    files = []

    for folder in SEARCH_DIRS:
        root = PROJECT_ROOT / folder
        if root.exists():
            files.extend(root.rglob("*"))

    parsed = []

    for f in files:
        if f.is_file():
            info = parse_file(f)
            if info:
                parsed.append(info)

    return parsed


def build_destination(info):
    sim_part = (
        f"sims_{info['sim_count']}"
        if info["sim_count"] is not None
        else "all_sims"
    )

    darcy_part = (
        f"darcy_{info['darcy_weight']}"
        if info["darcy_weight"] is not None
        else "darcy_unknown"
    )

    folder = (
        DEST_ROOT
        / info["group"]
        / info["training_mode"]
        / info["dataset_mode"]
        / info["density"]
        / info["model_type"]
        / sim_part
        / darcy_part
    )

    filename = f"{info['file_kind']}{info['extension']}"

    return folder / filename


def organize_models(parsed):
    manifest_rows = []

    for info in parsed:
        src = info["source_path"]
        dst = build_destination(info)

        manifest_rows.append({
            **{k: v for k, v in info.items() if k != "source_path"},
            "source_path": str(src),
            "dest_path": str(dst),
        })

        print(f"{src} -> {dst}")

        if not DRY_RUN:
            dst.parent.mkdir(parents=True, exist_ok=True)

            if dst.exists():
                print(f"  SKIP: destination exists")
                continue

            if COPY_INSTEAD_OF_MOVE:
                shutil.copy2(src, dst)
            else:
                shutil.move(src, dst)

    manifest_path = DEST_ROOT / "model_manifest.csv"

    if not DRY_RUN:
        DEST_ROOT.mkdir(parents=True, exist_ok=True)

        with open(manifest_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=manifest_rows[0].keys())
            writer.writeheader()
            writer.writerows(manifest_rows)

    print(f"\nManifest saved to: {manifest_path}")


# =========================
# PART 2: CHECK GAPS
# =========================

def existing_keys(parsed):
    keys = set()

    for info in parsed:
        keys.add((
            info["group"],
            info["training_mode"],
            info["dataset_mode"],
            info["density"],
            info["model_type"],
            info["sim_count"],
            info["darcy_weight"],
            info["file_kind"],
        ))

    return keys


def check_expected_models(parsed):
    keys = existing_keys(parsed)
    missing = []

    # Official Darcy expectations
    for dataset, model, weight, kind in product(
        ["fixed", "border"],
        MODEL_TYPES,
        ["0p1", "1p0", "5p0", "10p0"],
        FILE_KINDS,
    ):
        expected = (
            "official_darcy",
            "physics_limited",
            dataset,
            "unknown",
            model,
            None,
            weight,
            kind,
        )

        if expected not in keys:
            missing.append(expected)

    # Baseline expectations
    for dataset, model, kind in product(
        ["fixed", "border"],
        MODEL_TYPES,
        FILE_KINDS,
    ):
        expected = (
            "baseline_full",
            "baseline_full",
            dataset,
            "unknown",
            model,
            None,
            "0p0",
            kind,
        )

        if expected not in keys:
            missing.append(expected)

    # Data amount expectations
    for model, sim_count, weight, kind in product(
        ["splitnet_attn", "attn_unet", "unetfixed"],
        [25, 50, 100],
        ["1p0", "10p0"],
        FILE_KINDS,
    ):
        expected = (
            "data_amount_tests",
            "physics_limited",
            "fixed",
            "unknown",
            model,
            sim_count,
            weight,
            kind,
        )

        if expected not in keys:
            missing.append(expected)

    for model, sim_count, kind in product(
        ["splitnet_attn", "attn_unet", "unetfixed"],
        [25, 50, 100],
        FILE_KINDS,
    ):
        expected = (
            "data_amount_tests",
            "baseline_full",
            "fixed",
            "unknown",
            model,
            sim_count,
            "0p0",
            kind,
        )

        if expected not in keys:
            missing.append(expected)

    missing_path = DEST_ROOT / "missing_models.csv"

    if not DRY_RUN:
        with open(missing_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "group",
                "training_mode",
                "dataset_mode",
                "density",
                "model_type",
                "sim_count",
                "darcy_weight",
                "file_kind",
            ])
            writer.writerows(missing)

    print(f"\nMissing combinations: {len(missing)}")
    print(f"Missing report saved to: {missing_path}")

    return missing


# =========================
# RUN
# =========================

if __name__ == "__main__":
    parsed = find_model_files()

    print(f"Found {len(parsed)} model-related files.")

    organize_models(parsed)
    missing = check_expected_models(parsed)

    print("\nDone.")

Run fist with 

COPY_INSTEAD_OF_MOVE = True
DRY_RUN = True

if printed paths look good then 
COPY_INSTEAD_OF_MOVE = True
DRY_RUN = False